<img src="https://raw.githubusercontent.com/tangshunpu/OSD-JSCC/main/assets/logo.webp" width="120" align="right">

# OSD-JSCC demo

Send an image over a simulated **AWGN** channel and compare **OSD-JSCC**
(one-step diffusion) with the **SwinJSCC** baseline. Both use the same
bandwidth: CR = 96, that is 8 complex channel symbols per 16×16 pixel block.

**Before you start:** *Runtime → Change runtime type → T4 GPU*, then run the
cells in order (or *Runtime → Run all*). The first run downloads SD-Turbo
(3.6 GB) and the two checkpoints (420 MB), which takes a few minutes.

Code and paper: [github.com/tangshunpu/OSD-JSCC](https://github.com/tangshunpu/OSD-JSCC)

In [ ]:
#@title 1. Set up
import os
import subprocess


def sh(command):
    subprocess.run(command, shell=True, check=True)


if not os.path.isdir("OSD-JSCC"):
    # Code only: the weights come from the Hugging Face Hub in the next cell.
    sh("GIT_LFS_SKIP_SMUDGE=1 git clone -q --depth 1 https://github.com/tangshunpu/OSD-JSCC.git")
if not os.path.isdir("TF_SemCom"):
    # SwinJSCC (WITT) code, pinned to the commit used for the paper's baseline.
    sh("git clone -q https://github.com/tangshunpu/TF_SemCom.git")
    sh("git -C TF_SemCom checkout -q 75a581df18d19c34b078d0062f8769be0012d3ed")

# Colab ships an old torchao that recent peft refuses to load; OSD-JSCC does not use it.
%pip uninstall -q -y torchao
%pip install -q diffusers peft pyiqa timm

In [ ]:
#@title 2. Load OSD-JSCC and SwinJSCC
import contextlib
import io
import sys
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download

sys.path[:0] = ["OSD-JSCC", "TF_SemCom"]
from inference import Metrics, TO_TENSOR, load_model, seed_everything, to_uint8, transmit
from baselines.WITT.net.network import WITT

assert torch.cuda.is_available(), "Select a GPU: Runtime > Change runtime type > T4 GPU."
device = torch.device("cuda")
HF_REPO = "tangshunpu/OSD-JSCC"

# OSD-JSCC: SD-Turbo from the Hub, with the trained weights loaded on top.
osd = load_model(Path(hf_hub_download(HF_REPO, "osd_jscc_cr96_awgn.pt")), device, amp=True)
# Tile the VAE for images larger than 1536 px so they fit in a 16 GB T4.
osd.vae.enable_tiling()
osd.vae.tile_sample_min_size = 1536
osd.vae.tile_latent_min_size = 1536 // 8


def witt_config(C):
    """WITT configuration of TF_SemCom/baselines/WITT/witt_test.py."""
    blocks = dict(img_size=(256, 256), C=C, window_size=8, mlp_ratio=4.0, qkv_bias=True,
                  qk_scale=None, norm_layer=nn.LayerNorm, patch_norm=True)
    return dict(
        downsample=4,
        encoder_kwargs=dict(patch_size=2, in_chans=3, embed_dims=[128, 192, 256, 320],
                            depths=[2, 2, 6, 2], num_heads=[4, 6, 8, 10], **blocks),
        decoder_kwargs=dict(embed_dims=[320, 256, 192, 128], depths=[2, 6, 2, 2],
                            num_heads=[10, 8, 6, 4], **blocks),
    )


# SwinJSCC baseline of the paper: WITT with C = 16, trained on DIV2K at 0-20 dB.
swin_ckpt = torch.load(hf_hub_download(HF_REPO, "swinjscc_witt_c16_awgn.pth"),
                       map_location="cpu", weights_only=True)
with contextlib.redirect_stdout(io.StringIO()):  # WITT prints its layer sizes
    swin = WITT(witt_config(swin_ckpt["config"]["C"]), channel_type="AWGN").to(device).eval()
swin.load_state_dict(swin_ckpt["model_state_dict"])


@torch.inference_mode()
def transmit_swinjscc(image, snr_db):
    """As in the paper's evaluation: fp32, reflect-padded to a multiple of 256."""
    height, width = image.shape[-2:]
    padded = F.pad(image, (0, -width % 256, 0, -height % 256), mode="reflect")
    reconstruction, _, _ = swin(padded, given_SNR=snr_db)
    return reconstruction[..., :height, :width]


def compare(snr_db, seed):
    """Transmit the current image with both systems at one SNR."""
    seed_everything(seed)
    swin_out = transmit_swinjscc(image, snr_db)
    seed_everything(seed)
    osd_out = transmit(osd, image, snr_db, amp=True)
    return {"SwinJSCC": swin_out, "OSD-JSCC (ours)": osd_out}


def to_array(tensor):
    return to_uint8(tensor[0]).permute(1, 2, 0).cpu().numpy()


metrics = Metrics(["psnr", "lpips", "dists"], device)
print("Models ready.")

In [ ]:
#@title 3. Choose an image
image_name = "kodim23"  #@param ["kodim23", "kodim05", "kodim14", "kodim21", "upload"]
#@markdown Larger images are downscaled so that their long side is at most `max_side`.
max_side = 2048  #@param {type:"integer"}

import urllib.request

from PIL import Image

if image_name == "upload":
    from google.colab import files

    path = Path(next(iter(files.upload())))
else:
    path = Path(f"{image_name}.png")
    if not path.exists():
        urllib.request.urlretrieve(f"https://r0k.us/graphics/kodak/kodak/{image_name}.png", path)

picture = Image.open(path).convert("RGB")
if max(picture.size) > max_side:
    picture.thumbnail((max_side, max_side), Image.Resampling.LANCZOS)
image = TO_TENSOR(picture).unsqueeze(0).to(device)
print(f"{path.name}: {picture.width}x{picture.height}")

In [ ]:
#@title 4. Transmit at one SNR and compare
snr_db = 0  #@param {type:"slider", min:0, max:20, step:1}
seed = 42  #@param {type:"integer"}
#@markdown Zoomed region, as fractions of the image width and height:
zoom_x = 0.7  #@param {type:"slider", min:0, max:1, step:0.05}
zoom_y = 0.3  #@param {type:"slider", min:0, max:1, step:0.05}
zoom_size = 0.3  #@param {type:"slider", min:0.05, max:1, step:0.05}

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

outputs = compare(snr_db, seed)
panels = {"Original": to_array(image), **{name: to_array(out) for name, out in outputs.items()}}
height, width = panels["Original"].shape[:2]
side = max(16, int(zoom_size * min(height, width)))
top, left = int(zoom_y * (height - side)), int(zoom_x * (width - side))

fig, axes = plt.subplots(2, 3, figsize=(15, 5 * height / width + 5.4))
for column, (name, array) in enumerate(panels.items()):
    if name == "Original":
        title = f"Original ({width}x{height})"
        axes[0, column].add_patch(Rectangle((left, top), side, side, fill=False, ec="yellow", lw=2))
    else:
        score = metrics(outputs[name], image)
        title = (f"{name}\nPSNR {score['psnr']:.2f} dB | LPIPS {score['lpips']:.3f}"
                 f" | DISTS {score['dists']:.3f}")
    color = "#D62728" if "ours" in name else "black"
    axes[0, column].imshow(array)
    axes[0, column].set_title(title, color=color, fontsize=11)
    axes[1, column].imshow(array[top:top + side, left:left + side])
    for row in (0, 1):
        axes[row, column].axis("off")
fig.suptitle(f"AWGN, SNR = {snr_db} dB, CR = 96", fontsize=13)
fig.tight_layout()
plt.show()

os.makedirs("outputs", exist_ok=True)
for name, out in outputs.items():
    tag = "osd_jscc" if "OSD" in name else "swinjscc"
    Image.fromarray(to_array(out)).save(f"outputs/{path.stem}_{tag}_{snr_db}dB.png")
print("Saved the reconstructions to outputs/ (Files panel on the left).")

In [ ]:
#@title 5. Compare across SNRs (uses the zoomed region from step 4)
snr_list = "0, 5, 10, 15, 20"  #@param {type:"string"}

snrs = [float(value) for value in snr_list.split(",")]
names = ["SwinJSCC", "OSD-JSCC (ours)"]
crops, scores = {}, {name: [] for name in names}
for snr in snrs:
    for name, out in compare(snr, seed).items():
        crops[name, snr] = to_array(out)[top:top + side, left:left + side]
        scores[name].append(metrics(out, image))

fig, axes = plt.subplots(2, len(snrs), figsize=(3 * len(snrs), 6.4), squeeze=False)
for column, snr in enumerate(snrs):
    for row, name in enumerate(names):
        axes[row, column].imshow(crops[name, snr])
        axes[row, column].set_title(f"{snr:g} dB", fontsize=11)
        axes[row, column].set_xticks([])
        axes[row, column].set_yticks([])
    axes[0, 0].set_ylabel("SwinJSCC", fontsize=12)
    axes[1, 0].set_ylabel("OSD-JSCC (ours)", fontsize=12, color="#D62728")
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for axis, (key, label) in zip(axes, [("psnr", "PSNR (dB) ↑"), ("lpips", "LPIPS ↓"), ("dists", "DISTS ↓")]):
    for name in names:
        style = dict(color="#D62728", marker="s") if "ours" in name else dict(color="#4CB7A5", marker="o")
        axis.plot(snrs, [s[key] for s in scores[name]], label=name, **style)
    axis.set_xlabel("SNR (dB)")
    axis.set_ylabel(label)
    axis.grid(alpha=0.3)
axes[0].legend()
fig.tight_layout()
plt.show()

The LPIPS and DISTS numbers above are for this one image. The paper's results
over Kodak and CLIC2020 are in the
[README](https://github.com/tangshunpu/OSD-JSCC#-quantitative-results). To run
OSD-JSCC on a whole folder, use `inference.py` from the repository.